# Transcript Theme-Relevance Analyzer — Colab / Kaggle runner

This notebook is a **self-contained** copy of the `transcript_theme_analyzer`
package (see the `%%writefile` cells below) plus everything needed to run it
here: install dependencies, provide your API key, provide transcripts, run
the analysis, and view the results.

> **Before you run this: a compute-expectation note.** This pipeline is
> **I/O-bound** — every step is a network call to an LLM API (OpenAI or
> OpenRouter). There's no local heavy computation, so Colab/Kaggle's GPU/CPU
> power won't make it faster; the bottleneck is the LLM provider's response
> time and any rate limits, not your local machine. What this notebook *does*
> give you: a free hosted environment, zero local Python/venv setup, and
> convenient built-in secrets management for your API key.

> **Keeping this in sync:** the code cells below are embedded copies of the
> real package files. If you keep editing the code locally afterward, you'll
> need to regenerate this notebook (or manually copy the updated files back
> in) — it won't auto-update.


In [ ]:
import os

os.makedirs("transcript_theme_analyzer", exist_ok=True)
print("Created transcript_theme_analyzer/")


In [ ]:
%%writefile transcript_theme_analyzer/__init__.py


In [ ]:
%%writefile transcript_theme_analyzer/config.py
"""Environment-driven configuration for the analyzer pipeline."""
from __future__ import annotations

import os

from pydantic import BaseModel, Field


def _load_dotenv() -> None:
    """Minimal .env loader so the package has no extra dependency."""
    path = os.path.join(os.getcwd(), ".env")
    if not os.path.isfile(path):
        return
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, value = line.partition("=")
            key = key.strip()
            value = value.strip().strip('"').strip("'")
            os.environ.setdefault(key, value)


_load_dotenv()


class Config(BaseModel):
    provider: str = Field(default_factory=lambda: os.environ.get("LLM_PROVIDER", "openrouter"))
    base_url: str = Field(
        default_factory=lambda: os.environ.get(
            "LLM_BASE_URL", "https://openrouter.ai/api/v1"
        )
    )
    api_key: str = Field(
        default_factory=lambda: os.environ.get("OPENROUTER_API_KEY")
        or os.environ.get("OPENAI_API_KEY")
        or ""
    )
    default_model: str = Field(
        default_factory=lambda: os.environ.get(
            "LLM_DEFAULT_MODEL", "anthropic/claude-sonnet-5"
        )
    )
    chunk_size_tokens: int = Field(
        default_factory=lambda: int(os.environ.get("CHUNK_SIZE_TOKENS", "12000"))
    )
    chunk_overlap_tokens: int = Field(
        default_factory=lambda: int(os.environ.get("CHUNK_OVERLAP_TOKENS", "800"))
    )
    single_pass_token_limit: int = Field(
        default_factory=lambda: int(os.environ.get("SINGLE_PASS_TOKEN_LIMIT", "20000"))
    )
    max_concurrent_chunks: int = Field(
        default_factory=lambda: int(os.environ.get("MAX_CONCURRENT_CHUNKS", "8"))
    )
    max_retries: int = Field(
        default_factory=lambda: int(os.environ.get("LLM_MAX_RETRIES", "5"))
    )
    max_output_tokens: int = Field(
        default_factory=lambda: int(os.environ.get("LLM_MAX_OUTPUT_TOKENS", "4096"))
    )


def load_config() -> Config:
    return Config()


In [ ]:
%%writefile transcript_theme_analyzer/client.py
"""Model-agnostic OpenAI SDK client factory.

Works against OpenAI directly or any OpenAI-compatible endpoint (e.g.
OpenRouter) purely via `base_url` / `api_key` config -- no code changes
needed to swap providers.
"""
from __future__ import annotations

from openai import AsyncOpenAI

from .config import Config


def make_client(config: Config) -> AsyncOpenAI:
    return AsyncOpenAI(base_url=config.base_url, api_key=config.api_key)


In [ ]:
%%writefile transcript_theme_analyzer/chunker.py
"""Splits a transcript into overlapping chunks with positional metadata.

Token counts are approximated with a chars-per-token heuristic rather than a
real tokenizer, since this pipeline is model-agnostic (OpenAI tokenizers don't
apply to every provider/model it might run against anyway). The heuristic is
conservative (biased toward smaller chunks) so real token counts stay under
the configured budget.
"""
from __future__ import annotations

import re
from dataclasses import dataclass

CHARS_PER_TOKEN = 3.5

# Matches common transcript timestamp styles: [00:12:34], (00:12:34), 00:12:34.
TIMESTAMP_RE = re.compile(r"[\[\(]?\b\d{1,2}:\d{2}(?::\d{2})?\b[\]\)]?")
# Matches "Speaker Name:" / "SPEAKER 1:" at the start of a line.
SPEAKER_RE = re.compile(
    r"^\s*(?:[\[\(]?\d{1,2}:\d{2}(?::\d{2})?[\]\)]?\s*)?([A-Za-z][A-Za-z0-9 ._'-]{0,40}):\s",
    re.MULTILINE,
)


@dataclass
class Chunk:
    index: int
    char_start: int
    char_end: int
    text: str
    nearest_timestamp: str | None
    nearest_speaker: str | None


def _tokens_to_chars(n_tokens: int) -> int:
    return int(n_tokens * CHARS_PER_TOKEN)


def has_structure(transcript: str, sample_size: int = 20000) -> tuple[bool, bool]:
    """Detect whether the transcript carries timestamps and/or speaker labels."""
    sample = transcript[:sample_size]
    has_timestamps = bool(TIMESTAMP_RE.search(sample))
    has_speakers = bool(SPEAKER_RE.search(sample))
    return has_timestamps, has_speakers


def _nearest_match(pattern: re.Pattern, text: str, pos: int) -> str | None:
    """Find the match of `pattern` closest to (at or before) `pos`; else the first after it."""
    best_before = None
    for m in pattern.finditer(text[: pos + 1]):
        best_before = m
    if best_before is not None:
        return best_before.group(1) if pattern is SPEAKER_RE else best_before.group(0)
    after = pattern.search(text, pos)
    if after is not None:
        return after.group(1) if pattern is SPEAKER_RE else after.group(0)
    return None


def chunk_transcript(
    transcript: str,
    chunk_size_tokens: int,
    overlap_tokens: int,
) -> list[Chunk]:
    """Split `transcript` into overlapping chunks.

    Splits on paragraph/line boundaries near the target size where possible,
    so a chunk doesn't cut mid-sentence any more than necessary.
    """
    chunk_chars = _tokens_to_chars(chunk_size_tokens)
    overlap_chars = _tokens_to_chars(overlap_tokens)
    if overlap_chars >= chunk_chars:
        raise ValueError("chunk_overlap_tokens must be smaller than chunk_size_tokens")

    has_ts, has_sp = has_structure(transcript)
    n = len(transcript)
    chunks: list[Chunk] = []
    start = 0
    index = 0
    while start < n:
        end = min(start + chunk_chars, n)
        if end < n:
            # Prefer to break at a paragraph or line boundary within the last 20%
            search_from = max(start, end - int(chunk_chars * 0.2))
            boundary = transcript.rfind("\n", search_from, end)
            if boundary != -1 and boundary > start:
                end = boundary + 1

        text = transcript[start:end]
        nearest_ts = _nearest_match(TIMESTAMP_RE, transcript, start) if has_ts else None
        nearest_sp = _nearest_match(SPEAKER_RE, transcript, start) if has_sp else None

        chunks.append(
            Chunk(
                index=index,
                char_start=start,
                char_end=end,
                text=text,
                nearest_timestamp=nearest_ts,
                nearest_speaker=nearest_sp,
            )
        )

        if end >= n:
            break
        start = max(end - overlap_chars, start + 1)
        index += 1

    return chunks


def estimate_tokens(text: str) -> int:
    return int(len(text) / CHARS_PER_TOKEN)


In [ ]:
%%writefile transcript_theme_analyzer/schema.py
"""Pydantic models for the analysis output schema."""
from __future__ import annotations

from typing import Optional

from pydantic import BaseModel, Field


class Location(BaseModel):
    excerpt: str = Field(description="Short verbatim quote from the transcript, enough to locate it")
    context_summary: str = Field(description="1-sentence description of what's happening here re: the theme")
    char_start: int
    char_end: int
    timestamp: Optional[str] = None
    speaker: Optional[str] = None


class AnalysisResult(BaseModel):
    """Final, top-level result returned by the pipeline for one (transcript, theme, model) run."""

    theme: str
    relevance_score: int = Field(ge=0, le=100)
    reasoning: str
    locations: list[Location] = Field(default_factory=list)
    model_used: str
    chunked: bool


class ChunkAnalysis(BaseModel):
    """Partial result produced by the map step, for a single chunk."""

    relevance_score: int = Field(ge=0, le=100)
    explicitness: str = Field(
        description="How direct the theme's presence in this chunk is: 'explicit', 'tangential', or 'absent'"
    )
    reasoning: str
    locations: list[Location] = Field(default_factory=list)


In [ ]:
%%writefile transcript_theme_analyzer/aggregate.py
"""Reduce step: merges chunk-level analyses into one final AnalysisResult.

The final relevance_score is an explicit, tunable heuristic (not a naive
max/average of chunk scores): it combines
  - peak intensity   (the single most relevant chunk -- catches a strong,
                       concentrated discussion that would get diluted by
                       averaging over a long transcript)
  - average intensity (weighted by chunk length -- rewards sustained
                       discussion over one spike)
  - coverage breadth (what fraction of the transcript, by length, discusses
                       the theme at all, weighted double for "explicit" vs
                       "tangential" chunks)

Weights are keyword arguments so this can be tuned, or swapped for an LLM
aggregation pass later, without touching the map step.
"""
from __future__ import annotations

from .schema import ChunkAnalysis, Location

DEFAULT_WEIGHTS = {
    "peak": 0.30,
    "average": 0.40,
    "coverage": 0.30,
}

EXPLICITNESS_COVERAGE_WEIGHT = {
    "explicit": 1.0,
    "tangential": 0.5,
    "absent": 0.0,
}


def compute_aggregate_score(
    chunk_results: list[ChunkAnalysis],
    chunk_lengths: list[int],
    weights: dict[str, float] | None = None,
) -> int:
    if not chunk_results:
        return 0
    weights = weights or DEFAULT_WEIGHTS
    total_length = sum(chunk_lengths) or 1

    peak = max(c.relevance_score for c in chunk_results)

    weighted_sum = sum(c.relevance_score * length for c, length in zip(chunk_results, chunk_lengths))
    average = weighted_sum / total_length

    coverage_weighted_length = sum(
        length * EXPLICITNESS_COVERAGE_WEIGHT.get(c.explicitness, 0.0)
        for c, length in zip(chunk_results, chunk_lengths)
    )
    coverage = (coverage_weighted_length / total_length) * 100

    final = weights["peak"] * peak + weights["average"] * average + weights["coverage"] * coverage
    return max(0, min(100, round(final)))


def merge_and_dedupe_locations(
    all_locations: list[Location], overlap_threshold: float = 0.5
) -> list[Location]:
    """Merge locations from overlapping chunk boundaries.

    Locations are already translated to full-transcript char offsets by the
    caller. Two locations are treated as duplicates of the same mention if
    their char ranges overlap by more than `overlap_threshold` of the smaller
    range's length -- this is what happens when the same passage falls in the
    overlap region of two adjacent chunks and both flag it.
    """
    if not all_locations:
        return []

    ordered = sorted(all_locations, key=lambda loc: (loc.char_start, loc.char_end))
    kept: list[Location] = [ordered[0]]

    for loc in ordered[1:]:
        prev = kept[-1]
        overlap = min(loc.char_end, prev.char_end) - max(loc.char_start, prev.char_start)
        smaller_len = min(loc.char_end - loc.char_start, prev.char_end - prev.char_start) or 1
        if overlap > 0 and (overlap / smaller_len) > overlap_threshold:
            # Keep the one with the longer excerpt (usually the more complete capture).
            if len(loc.excerpt) > len(prev.excerpt):
                kept[-1] = loc
            continue
        kept.append(loc)

    return kept


In [ ]:
%%writefile transcript_theme_analyzer/prompts.py
"""Versioned system prompts for the analysis and synthesis LLM calls."""

CHUNK_ANALYSIS_SYSTEM_PROMPT_V1 = """\
You are a careful analyst assessing how thoroughly a TRANSCRIPT SEGMENT discusses a given THEME.

The theme describes a concept, not a literal string to search for. Reason about it \
semantically: a segment can be highly relevant to the theme without ever using the \
theme's exact words, and can mention the theme's words without being substantively \
about it.

For every location you report, be explicit about whether the connection is:
- "explicit": the segment directly and substantively discusses the theme
- "tangential": the segment touches on the theme only in passing, implicitly, or as a \
minor aside relative to its main subject

Factor this distinction into both your relevance_score and your reasoning. A segment \
dense with explicit discussion should score much higher than one with only tangential \
references, even if both tYes echnically "mention" the theme.

Rules you must follow:
1. Never fabricate excerpts. Every "excerpt" field must be a verbatim substring (or a \
very close paraphrase clearly traceable to actual text) of the segment you were given. \
Do not invent quotes that sound plausible but are not actually present.
2. If the theme does not meaningfully appear in this segment at all, say so plainly: \
score near 0, return an empty (or near-empty) locations list, and give an honest, \
brief reasoning. Do not strain to invent a weak connection just to have something to \
report.
3. `char_start` / `char_end` must be offsets into the segment text you were given, \
measured from the start of that segment (not the full transcript) -- the caller will \
translate these into full-transcript offsets.
4. If the segment includes embedded timestamps or speaker labels, capture the ones \
nearest each location you report in the `timestamp` / `speaker` fields. If the segment \
has no such markers, omit those fields.
5. Set `explicitness` to the single best characterization of the segment's overall \
relationship to the theme: "explicit", "tangential", or "absent".
6. Output only valid JSON matching the required schema. Do not include any prose \
outside the JSON structure.

You are scoring one segment of a much larger transcript in isolation. Do not assume \
context you were not given, and do not penalize the segment for not covering the whole \
theme -- your job is to describe what this segment itself contains.
"""

SYNTHESIS_SYSTEM_PROMPT_V1 = """\
You are synthesizing several partial analyses of different segments of one transcript \
into a single final reasoning paragraph about how thoroughly the transcript as a whole \
discusses a given theme.

You will be given the theme, the transcript's overall relevance score (already \
computed), and a list of per-segment partial reasonings with their scores and \
explicitness ratings, in transcript order.

Write one coherent final `reasoning` that:
- States how central vs. incidental the theme was overall.
- Notes whether the discussion was concentrated in one part of the transcript or \
distributed throughout.
- Explains, briefly, why the score isn't higher or lower -- referencing the pattern \
across segments rather than re-listing every segment individually.
- Is written as a single piece of prose for a human reader, not a concatenation or \
bullet list of the inputs you were given.

Do not introduce new claims about the transcript's content beyond what the partial \
analyses support. Output only valid JSON matching the required schema.
"""

SINGLE_PASS_USER_PROMPT_TEMPLATE = """\
THEME:
{theme}

TRANSCRIPT:
{transcript}
"""

CHUNK_USER_PROMPT_TEMPLATE = """\
THEME:
{theme}

TRANSCRIPT SEGMENT (chunk {chunk_index} of {total_chunks}, characters {char_start}-{char_end} \
of the full transcript):
{chunk_text}
"""


In [ ]:
%%writefile transcript_theme_analyzer/analyzer.py
"""Core analysis pipeline: single-pass fast path + chunked map-reduce path."""
from __future__ import annotations

import asyncio
import json
import logging
import random
from dataclasses import dataclass, field

import openai
from openai import AsyncOpenAI
from pydantic import BaseModel

from . import prompts
from .aggregate import compute_aggregate_score, merge_and_dedupe_locations
from .chunker import chunk_transcript, estimate_tokens
from .config import Config
from .schema import AnalysisResult, ChunkAnalysis, Location

logger = logging.getLogger("transcript_theme_analyzer")


@dataclass
class CallLog:
    label: str
    model: str
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0


@dataclass
class RunStats:
    calls: list[CallLog] = field(default_factory=list)

    def add(self, label: str, model: str, usage) -> None:
        if usage is None:
            self.calls.append(CallLog(label=label, model=model))
            return
        self.calls.append(
            CallLog(
                label=label,
                model=model,
                prompt_tokens=getattr(usage, "prompt_tokens", 0) or 0,
                completion_tokens=getattr(usage, "completion_tokens", 0) or 0,
                total_tokens=getattr(usage, "total_tokens", 0) or 0,
            )
        )

    @property
    def total_tokens(self) -> int:
        return sum(c.total_tokens for c in self.calls)

    def summary(self) -> dict:
        return {
            "calls": len(self.calls),
            "total_tokens": self.total_tokens,
            "by_call": [c.__dict__ for c in self.calls],
        }


RETRYABLE_EXCEPTIONS = (
    openai.RateLimitError,
    openai.APIConnectionError,
    openai.InternalServerError,
    openai.APITimeoutError,
)


def _json_schema_for(model_cls, name: str) -> dict:
    schema = model_cls.model_json_schema()
    schema["additionalProperties"] = False
    return {"name": name, "schema": schema, "strict": True}


async def _call_with_retry(
    client: AsyncOpenAI,
    *,
    model: str,
    system: str,
    user: str,
    schema_cls,
    schema_name: str,
    max_retries: int,
    max_output_tokens: int,
):
    """Call the chat completions API, enforcing the schema.

    Tries native structured output (response_format=json_schema) first; if
    the provider/model rejects that parameter, falls back to a plain call
    with a strict parse-and-retry loop.
    """
    last_exc: Exception | None = None
    for attempt in range(max_retries):
        try:
            try:
                resp = await client.chat.completions.create(
                    model=model,
                    max_tokens=max_output_tokens,
                    messages=[
                        {"role": "system", "content": system},
                        {"role": "user", "content": user},
                    ],
                    response_format={
                        "type": "json_schema",
                        "json_schema": _json_schema_for(schema_cls, schema_name),
                    },
                )
            except openai.BadRequestError:
                # Provider doesn't support structured output mode -- fall back
                # to plain JSON-mode prompting and a strict parse retry below.
                resp = await client.chat.completions.create(
                    model=model,
                    max_tokens=max_output_tokens,
                    messages=[
                        {"role": "system", "content": system},
                        {
                            "role": "user",
                            "content": user
                            + "\n\nRespond with ONLY a single valid JSON object matching "
                            "the required schema. No markdown fences, no commentary.",
                        },
                    ],
                )

            content = resp.choices[0].message.content or ""
            content = content.strip()
            if content.startswith("```"):
                content = content.strip("`")
                if content.startswith("json"):
                    content = content[4:]
            parsed = json.loads(content)
            result = schema_cls.model_validate(parsed)
            return result, resp.usage
        except RETRYABLE_EXCEPTIONS as exc:
            last_exc = exc
            delay = min(2 ** attempt + random.uniform(0, 1), 30)
            logger.warning("Retryable error on attempt %d/%d: %s. Sleeping %.1fs", attempt + 1, max_retries, exc, delay)
            await asyncio.sleep(delay)
        except openai.APIStatusError:
            # Non-retryable API error (bad request, auth, billing/quota, not
            # found, etc.) -- retrying identical input against these just
            # burns the whole budget for a guaranteed-identical failure.
            raise
        except (json.JSONDecodeError, Exception) as exc:  # noqa: BLE001 - parse/validation retry
            last_exc = exc
            logger.warning("Parse/validation error on attempt %d/%d: %s", attempt + 1, max_retries, exc)
            await asyncio.sleep(min(1.5 ** attempt, 10))

    raise RuntimeError(f"Failed after {max_retries} attempts: {last_exc}") from last_exc


async def analyze_single_pass(
    client: AsyncOpenAI,
    model: str,
    theme: str,
    transcript: str,
    config: Config,
    stats: RunStats,
) -> AnalysisResult:
    user = prompts.SINGLE_PASS_USER_PROMPT_TEMPLATE.format(theme=theme, transcript=transcript)
    result, usage = await _call_with_retry(
        client,
        model=model,
        system=prompts.CHUNK_ANALYSIS_SYSTEM_PROMPT_V1,
        user=user,
        schema_cls=ChunkAnalysis,
        schema_name="chunk_analysis",
        max_retries=config.max_retries,
        max_output_tokens=config.max_output_tokens,
    )
    stats.add("single_pass", model, usage)

    locations = [
        Location(
            excerpt=loc.excerpt,
            context_summary=loc.context_summary,
            char_start=loc.char_start,
            char_end=loc.char_end,
            timestamp=loc.timestamp,
            speaker=loc.speaker,
        )
        for loc in result.locations
    ]

    return AnalysisResult(
        theme=theme,
        relevance_score=result.relevance_score,
        reasoning=result.reasoning,
        locations=locations,
        model_used=model,
        chunked=False,
    )


async def _analyze_one_chunk(
    client: AsyncOpenAI,
    model: str,
    theme: str,
    chunk,
    total_chunks: int,
    config: Config,
    stats: RunStats,
    semaphore: asyncio.Semaphore,
) -> tuple[ChunkAnalysis, "object"]:
    async with semaphore:
        user = prompts.CHUNK_USER_PROMPT_TEMPLATE.format(
            theme=theme,
            chunk_index=chunk.index + 1,
            total_chunks=total_chunks,
            char_start=chunk.char_start,
            char_end=chunk.char_end,
            chunk_text=chunk.text,
        )
        result, usage = await _call_with_retry(
            client,
            model=model,
            system=prompts.CHUNK_ANALYSIS_SYSTEM_PROMPT_V1,
            user=user,
            schema_cls=ChunkAnalysis,
            schema_name="chunk_analysis",
            max_retries=config.max_retries,
            max_output_tokens=config.max_output_tokens,
        )
        stats.add(f"chunk_{chunk.index}", model, usage)
        return result, chunk


async def _synthesize_reasoning(
    client: AsyncOpenAI,
    model: str,
    theme: str,
    final_score: int,
    chunk_results: list[tuple[ChunkAnalysis, object]],
    config: Config,
    stats: RunStats,
) -> str:
    lines = []
    for result, chunk in sorted(chunk_results, key=lambda pair: pair[1].index):
        lines.append(
            f"- Segment {chunk.index + 1} (chars {chunk.char_start}-{chunk.char_end}, "
            f"score {result.relevance_score}, {result.explicitness}): {result.reasoning}"
        )
    user = (
        f"THEME:\n{theme}\n\n"
        f"OVERALL COMPUTED SCORE: {final_score}\n\n"
        f"PER-SEGMENT PARTIAL ANALYSES (in transcript order):\n" + "\n".join(lines)
    )

    class SynthesisOutput(BaseModel):
        reasoning: str

    result, usage = await _call_with_retry(
        client,
        model=model,
        system=prompts.SYNTHESIS_SYSTEM_PROMPT_V1,
        user=user,
        schema_cls=SynthesisOutput,
        schema_name="synthesis_output",
        max_retries=config.max_retries,
        max_output_tokens=config.max_output_tokens,
    )
    stats.add("synthesis", model, usage)
    return result.reasoning


async def analyze_map_reduce(
    client: AsyncOpenAI,
    model: str,
    theme: str,
    transcript: str,
    config: Config,
    stats: RunStats,
) -> AnalysisResult:
    chunks = chunk_transcript(transcript, config.chunk_size_tokens, config.chunk_overlap_tokens)
    semaphore = asyncio.Semaphore(config.max_concurrent_chunks)

    tasks = [
        _analyze_one_chunk(client, model, theme, chunk, len(chunks), config, stats, semaphore)
        for chunk in chunks
    ]
    chunk_results = await asyncio.gather(*tasks)

    analyses = [r for r, _ in chunk_results]
    lengths = [c.char_end - c.char_start for _, c in chunk_results]
    final_score = compute_aggregate_score(analyses, lengths)

    all_locations: list[Location] = []
    for analysis, chunk in chunk_results:
        for loc in analysis.locations:
            all_locations.append(
                Location(
                    excerpt=loc.excerpt,
                    context_summary=loc.context_summary,
                    char_start=chunk.char_start + loc.char_start,
                    char_end=chunk.char_start + loc.char_end,
                    timestamp=loc.timestamp or chunk.nearest_timestamp,
                    speaker=loc.speaker or chunk.nearest_speaker,
                )
            )
    merged_locations = merge_and_dedupe_locations(all_locations)

    reasoning = await _synthesize_reasoning(
        client, model, theme, final_score, chunk_results, config, stats
    )

    return AnalysisResult(
        theme=theme,
        relevance_score=final_score,
        reasoning=reasoning,
        locations=merged_locations,
        model_used=model,
        chunked=True,
    )


async def analyze(
    client: AsyncOpenAI,
    model: str,
    theme: str,
    transcript: str,
    config: Config,
) -> tuple[AnalysisResult, RunStats]:
    """Entry point: picks the single-pass fast path or the chunked map-reduce path."""
    stats = RunStats()
    estimated_tokens = estimate_tokens(transcript)
    if estimated_tokens <= config.single_pass_token_limit:
        result = await analyze_single_pass(client, model, theme, transcript, config, stats)
    else:
        result = await analyze_map_reduce(client, model, theme, transcript, config, stats)
    return result, stats


In [ ]:
%%writefile transcript_theme_analyzer/loader.py
"""Detects a transcript input file's format and extracts plain text from it.

Supported formats: ``.txt`` (read as-is) and ``.docx`` (paragraph text via
python-docx). Rich formatting (bold/italic/underline, fonts, colors) is
intentionally dropped on extraction — nothing downstream (chunker.py's
timestamp/speaker regexes, the analysis prompts) reasons about typography,
only about the text content. Paragraph breaks ARE preserved (one paragraph
per line): chunk_transcript splits on "\\n" boundaries and SPEAKER_RE only
matches at the start of a line, so losing paragraph structure would break
both.
"""
from __future__ import annotations

import os
import zipfile

from docx import Document

SUPPORTED_EXTENSIONS = {".txt", ".docx"}


def detect_format(path: str) -> str:
    """Returns the lowercased extension (".txt" or ".docx"), or raises
    ValueError with a clear message if the file isn't a supported/valid
    transcript format."""
    ext = os.path.splitext(path)[1].lower()
    if ext not in SUPPORTED_EXTENSIONS:
        raise ValueError(
            f"Unsupported transcript file type {ext!r} for {path!r}; "
            f"supported: {', '.join(sorted(SUPPORTED_EXTENSIONS))}"
        )
    if ext == ".docx" and not zipfile.is_zipfile(path):
        raise ValueError(
            f"{path!r} has a .docx extension but isn't a valid .docx file (not a zip container)"
        )
    return ext


def _extract_docx_text(path: str) -> str:
    document = Document(path)
    lines = [p.text for p in document.paragraphs]

    # Collapse runs of 2+ blank paragraphs (common docx spacing) down to a
    # single blank line, so the chunker doesn't see artificially huge gaps.
    collapsed: list[str] = []
    for line in lines:
        if line == "" and collapsed and collapsed[-1] == "":
            continue
        collapsed.append(line)

    return "\n".join(collapsed).strip()


def load_transcript_text(path: str) -> str:
    ext = detect_format(path)
    if ext == ".txt":
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    return _extract_docx_text(path)


In [ ]:
%%writefile transcript_theme_analyzer/cache.py
"""Internal cache of raw per-run analysis payloads — not a user-facing
output. Lets ``report.py``/``word_report.py`` regenerate the html/docx
reports later without re-running the (paid) LLM analysis. Written as a
dot-prefixed file so it reads as internal, distinct from the two real
outputs (``report.html``, ``report.docx``).
"""
from __future__ import annotations

import json
import os

CACHE_FILENAME = ".raw_results.json"


def write_cache(theme: str, transcripts: dict[str, list[tuple[str, dict]]], out_dir: str) -> str:
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, CACHE_FILENAME)
    serializable = {
        "theme": theme,
        "results": {
            name: [{"model": model, "payload": payload} for model, payload in payloads]
            for name, payloads in transcripts.items()
        },
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(serializable, f, indent=2)
    return path


def read_cache(path: str) -> tuple[str, dict[str, list[tuple[str, dict]]]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    transcripts = {
        name: [(entry["model"], entry["payload"]) for entry in entries]
        for name, entries in data.get("results", {}).items()
    }
    return data.get("theme", "(unknown theme)"), transcripts


In [ ]:
%%writefile transcript_theme_analyzer/report.py
"""Shared results data model + the interactive HTML report.

``build_report`` turns the raw in-memory run payloads (as produced by
``cli.py``) into a ``BatchReport`` — the same structure ``word_report.py``
renders to docx. This module renders it to a single self-contained,
mobile-friendly, interactive ``report.html``.

Usage (regenerate later from the internal cache, without re-running analysis):
    python -m transcript_theme_analyzer.report --cache results/.raw_results.json
"""
from __future__ import annotations

import argparse
import os
import sys
from dataclasses import dataclass, field
from datetime import datetime, timezone
from html import escape as h

from .cache import read_cache


@dataclass
class ModelResult:
    model: str
    score: int | None = None
    reasoning: str = ""
    locations: list[dict] = field(default_factory=list)
    chunked: bool = False
    total_tokens: int | None = None
    elapsed_seconds: float | None = None
    error: str | None = None


@dataclass
class TranscriptResult:
    name: str
    models: list[ModelResult]

    @property
    def best_model(self) -> ModelResult | None:
        ok = [m for m in self.models if m.error is None and m.score is not None]
        if not ok:
            return None
        return max(ok, key=lambda m: m.score)

    @property
    def best_score(self) -> int | None:
        best = self.best_model
        return best.score if best else None


@dataclass
class BatchReport:
    theme: str
    transcripts: list[TranscriptResult]
    generated_at: str


def _model_result_from_data(model: str, data: dict) -> ModelResult:
    if "relevance_score" not in data:
        return ModelResult(model=model, error=data.get("error", "unknown error (malformed result)"))

    meta = data.get("_meta", {}) or {}
    usage = meta.get("usage", {}) or {}
    return ModelResult(
        model=model,
        score=data.get("relevance_score"),
        reasoning=data.get("reasoning", ""),
        locations=data.get("locations", []) or [],
        chunked=bool(data.get("chunked", False)),
        total_tokens=usage.get("total_tokens"),
        elapsed_seconds=meta.get("elapsed_seconds"),
    )


def build_report(theme: str, transcripts: dict[str, list[tuple[str, dict]]]) -> BatchReport:
    """Build a report directly from in-memory run payloads — one entry per
    transcript name, each a list of ``(model_name, payload_dict)`` pairs (as
    produced by ``cli.run_one_model``; a failed run's payload is
    ``{"model": ..., "error": ...}``).
    """
    transcript_results = [
        TranscriptResult(
            name=name,
            models=[_model_result_from_data(model, data) for model, data in model_payloads],
        )
        for name, model_payloads in transcripts.items()
    ]

    def sort_key(t: TranscriptResult):
        return (t.best_score is None, -(t.best_score or 0), t.name)

    transcript_results.sort(key=sort_key)
    return BatchReport(
        theme=theme,
        transcripts=transcript_results,
        generated_at=datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"),
    )


# --------------------------------------------------------------------------
# HTML
# --------------------------------------------------------------------------

def _bar_color_var(score: int) -> str:
    """Sequential single-hue (blue) bucket: low scores recede toward the
    surface, high scores get the strongest step — per the dataviz sequential
    convention (lightest = near zero)."""
    if score >= 70:
        return "var(--seq-700)"
    if score >= 40:
        return "var(--seq-500)"
    return "var(--seq-300)"


def _render_locations_html(locations: list[dict], limit: int) -> str:
    if not locations:
        return "<p class='muted'>No specific locations recorded.</p>"
    items = []
    for loc in locations[:limit]:
        ts = loc.get("timestamp")
        speaker = loc.get("speaker")
        meta_bits = [b for b in (ts, speaker) if b]
        meta_str = h(" · ".join(meta_bits)) if meta_bits else ""
        excerpt = h(loc.get("excerpt", "").replace("\n", " "))
        summary = h(loc.get("context_summary", ""))
        items.append(
            f"<li><span class='loc-meta'>{meta_str}</span>"
            f"<blockquote>&ldquo;{excerpt}&rdquo;</blockquote>"
            f"<span class='loc-summary'>{summary}</span></li>"
        )
    more = len(locations) - limit
    more_note = f"<p class='muted'>+{more} more location(s).</p>" if more > 0 else ""
    return f"<ul class='locations'>{''.join(items)}</ul>{more_note}"


def _render_transcript_row(rank: int, t: TranscriptResult) -> str:
    score = t.best_score
    score_display = str(score) if score is not None else "—"
    bar_width = score if score is not None else 0
    bar_color = _bar_color_var(score) if score is not None else "var(--muted)"
    status = "" if score is not None else "<span class='badge badge-error'>FAILED</span>"
    return f"""
    <tr class="rank-row" data-score="{score if score is not None else -1}" data-name="{h(t.name)}">
      <td class="col-rank" data-label="#">{rank}</td>
      <td class="col-name" data-label="Transcript"><a href="#detail-{rank}">{h(t.name)}</a> {status}</td>
      <td class="col-score" data-label="Score">{score_display}</td>
      <td class="col-bar" data-label="Relevance">
        <div class="bar-track">
          <div class="bar-fill" style="width:{bar_width}%; background:{bar_color};"></div>
        </div>
      </td>
    </tr>
    """


def _render_transcript_detail(rank: int, t: TranscriptResult, top_locations: int) -> str:
    best = t.best_model
    if best is None:
        errors = "".join(f"<li><strong>{h(m.model)}</strong>: {h(m.error or '')}</li>" for m in t.models)
        return f"""
        <details class="transcript-detail" id="detail-{rank}">
          <summary>{rank}. {h(t.name)} <span class="badge badge-error">FAILED</span></summary>
          <div class="detail-body">
            <p>All model runs failed for this transcript:</p>
            <ul>{errors}</ul>
          </div>
        </details>
        """

    other_models_html = ""
    if len(t.models) > 1:
        chips = "".join(
            f"<span class='chip'>{h(m.model)}: {m.score if m.error is None else 'error'}</span>"
            for m in t.models if m is not best
        )
        other_models_html = f"<div class='other-models'>{chips}</div>"

    meta_bits = [f"model: <code>{h(best.model)}</code>"]
    if best.chunked:
        meta_bits.append("chunked (map-reduce)")
    if best.total_tokens is not None:
        meta_bits.append(f"{best.total_tokens:,} tokens")
    if best.elapsed_seconds is not None:
        meta_bits.append(f"{best.elapsed_seconds:.1f}s")

    return f"""
    <details class="transcript-detail" id="detail-{rank}">
      <summary>{rank}. {h(t.name)} <span class="score-pill">{best.score}</span></summary>
      <div class="detail-body">
        <p class="run-meta">{' &middot; '.join(meta_bits)}</p>
        {other_models_html}
        <h4>Reasoning</h4>
        <p class="reasoning">{h(best.reasoning)}</p>
        <h4>Key locations</h4>
        {_render_locations_html(best.locations, top_locations)}
      </div>
    </details>
    """


def render_html(report: BatchReport, top_locations: int = 5) -> str:
    rows = "".join(_render_transcript_row(i, t) for i, t in enumerate(report.transcripts, start=1))
    details = "".join(_render_transcript_detail(i, t, top_locations) for i, t in enumerate(report.transcripts, start=1))
    n_ok = sum(1 for t in report.transcripts if t.best_score is not None)
    n_failed = len(report.transcripts) - n_ok

    return f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Theme Analysis: {h(report.theme)}</title>
<style>
  :root {{
    color-scheme: light;
    --surface-1: #fcfcfb;
    --page: #f9f9f7;
    --text-primary: #0b0b0b;
    --text-secondary: #52514e;
    --muted: #898781;
    --grid: #e1e0d9;
    --border: rgba(11,11,11,0.10);
    --seq-300: #6da7ec;
    --seq-500: #256abf;
    --seq-700: #0d366b;
    --critical: #d03b3b;
  }}
  @media (prefers-color-scheme: dark) {{
    :root:where(:not([data-theme="light"])) {{
      color-scheme: dark;
      --surface-1: #1a1a19;
      --page: #0d0d0d;
      --text-primary: #ffffff;
      --text-secondary: #c3c2b7;
      --muted: #898781;
      --grid: #2c2c2a;
      --border: rgba(255,255,255,0.10);
      --seq-300: #5598e7;
      --seq-500: #2a78d6;
      --seq-700: #3987e5;
      --critical: #e66767;
    }}
  }}
  :root[data-theme="dark"] {{
    color-scheme: dark;
    --surface-1: #1a1a19;
    --page: #0d0d0d;
    --text-primary: #ffffff;
    --text-secondary: #c3c2b7;
    --muted: #898781;
    --grid: #2c2c2a;
    --border: rgba(255,255,255,0.10);
    --seq-300: #5598e7;
    --seq-500: #2a78d6;
    --seq-700: #3987e5;
    --critical: #e66767;
  }}
  * {{ box-sizing: border-box; }}
  body {{
    margin: 0;
    padding: 2rem 1rem 4rem;
    background: var(--page);
    color: var(--text-primary);
    font-family: system-ui, -apple-system, "Segoe UI", sans-serif;
  }}
  .wrap {{ max-width: 900px; margin: 0 auto; }}
  h1 {{ font-size: 1.5rem; margin-bottom: 0.25rem; }}
  .subtitle {{ color: var(--text-secondary); margin-top: 0; }}
  .stats {{ display: flex; gap: 1.5rem; margin: 1.25rem 0; flex-wrap: wrap; }}
  .stat {{ background: var(--surface-1); border: 1px solid var(--border); border-radius: 8px; padding: 0.75rem 1rem; }}
  .stat .num {{ font-size: 1.4rem; font-weight: 600; display: block; }}
  .stat .label {{ font-size: 0.8rem; color: var(--text-secondary); }}
  .filter-bar {{ margin: 1.5rem 0 0.75rem; }}
  .filter-bar input {{
    width: 100%;
    font-size: 0.95rem;
    padding: 0.6rem 0.85rem;
    border-radius: 8px;
    border: 1px solid var(--border);
    background: var(--surface-1);
    color: var(--text-primary);
  }}
  .filter-bar input:focus {{ outline: 2px solid var(--seq-500); outline-offset: 1px; }}
  table {{ width: 100%; border-collapse: collapse; background: var(--surface-1); border: 1px solid var(--border); border-radius: 8px; overflow: hidden; }}
  th, td {{ padding: 0.6rem 0.75rem; text-align: left; border-bottom: 1px solid var(--grid); font-size: 0.9rem; }}
  th {{ color: var(--text-secondary); font-weight: 600; cursor: pointer; user-select: none; }}
  th:hover {{ color: var(--text-primary); }}
  th.sort-active::after {{ content: " \\25BC"; font-size: 0.7em; }}
  tr:last-child td {{ border-bottom: none; }}
  .col-rank {{ width: 3rem; color: var(--muted); }}
  .col-score {{ width: 4rem; font-variant-numeric: tabular-nums; text-align: right; }}
  .col-bar {{ width: 35%; }}
  .col-name a {{ color: inherit; text-decoration: none; border-bottom: 1px solid var(--border); }}
  .col-name a:hover {{ border-bottom-color: var(--seq-500); color: var(--seq-500); }}
  .bar-track {{ height: 10px; background: var(--grid); border-radius: 4px; overflow: hidden; }}
  .bar-fill {{ height: 100%; border-radius: 4px; }}
  .badge {{ font-size: 0.7rem; padding: 0.1rem 0.4rem; border-radius: 4px; margin-left: 0.4rem; }}
  .badge-error {{ background: var(--critical); color: #fff; }}
  .score-pill {{ font-variant-numeric: tabular-nums; background: var(--seq-500); color: #fff; border-radius: 999px; padding: 0.1rem 0.6rem; font-size: 0.85rem; margin-left: 0.5rem; }}
  .no-matches {{ text-align: center; color: var(--muted); padding: 1.5rem !important; }}
  h2 {{ font-size: 1.15rem; margin-top: 2.5rem; }}
  .transcript-detail {{ background: var(--surface-1); border: 1px solid var(--border); border-radius: 8px; margin-bottom: 0.6rem; padding: 0.6rem 1rem; scroll-margin-top: 1rem; }}
  .transcript-detail summary {{ cursor: pointer; font-weight: 600; padding: 0.4rem 0.1rem; }}
  .detail-body {{ margin-top: 0.75rem; }}
  .run-meta {{ color: var(--text-secondary); font-size: 0.85rem; }}
  .reasoning {{ line-height: 1.5; }}
  .locations {{ list-style: none; padding: 0; margin: 0; }}
  .locations li {{ padding: 0.5rem 0; border-top: 1px solid var(--grid); }}
  .locations li:first-child {{ border-top: none; }}
  .loc-meta {{ font-size: 0.75rem; color: var(--muted); display: block; margin-bottom: 0.2rem; }}
  blockquote {{ margin: 0.2rem 0; padding-left: 0.75rem; border-left: 2px solid var(--seq-500); color: var(--text-primary); }}
  .loc-summary {{ font-size: 0.85rem; color: var(--text-secondary); }}
  .chip {{ display: inline-block; background: var(--grid); border-radius: 999px; padding: 0.1rem 0.6rem; font-size: 0.8rem; margin: 0 0.3rem 0.3rem 0; }}
  .other-models {{ margin: 0.5rem 0; }}
  .muted {{ color: var(--muted); font-size: 0.85rem; }}
  code {{ background: var(--grid); border-radius: 4px; padding: 0.05rem 0.3rem; }}

  @media (max-width: 640px) {{
    body {{ padding: 1.25rem 0.75rem 3rem; }}
    .stats {{ gap: 0.6rem; }}
    .stat {{ flex: 1 1 40%; padding: 0.6rem 0.75rem; }}
    table, thead, tbody, th, td, tr {{ display: block; }}
    thead {{ position: absolute; left: -9999px; top: -9999px; }}
    tbody tr {{
      border: 1px solid var(--border);
      border-radius: 8px;
      margin-bottom: 0.6rem;
      padding: 0.35rem 0.75rem;
    }}
    tbody tr:last-child {{ margin-bottom: 0; }}
    td {{
      display: flex;
      justify-content: space-between;
      align-items: center;
      gap: 1rem;
      border-bottom: 1px solid var(--grid);
      padding: 0.5rem 0;
    }}
    td:last-child {{ border-bottom: none; }}
    td::before {{
      content: attr(data-label);
      font-weight: 600;
      color: var(--text-secondary);
      font-size: 0.8rem;
      flex-shrink: 0;
    }}
    .col-bar {{ flex-direction: column; align-items: stretch; }}
    .col-bar .bar-track {{ width: 100%; margin-top: 0.25rem; }}
    .col-name {{ text-align: right; }}
  }}
</style>
</head>
<body>
  <div class="wrap">
    <h1>Transcript Theme Analysis</h1>
    <p class="subtitle">Theme: &ldquo;{h(report.theme)}&rdquo; &middot; generated {h(report.generated_at)}</p>

    <div class="stats">
      <div class="stat"><span class="num">{len(report.transcripts)}</span><span class="label">transcripts</span></div>
      <div class="stat"><span class="num">{n_ok}</span><span class="label">scored</span></div>
      <div class="stat"><span class="num">{n_failed}</span><span class="label">failed</span></div>
    </div>

    <div class="filter-bar">
      <input type="search" id="transcript-filter" placeholder="Filter transcripts by name…" aria-label="Filter transcripts by name">
    </div>

    <table id="rank-table">
      <thead>
        <tr>
          <th data-sort="rank">#</th>
          <th data-sort="name">Transcript</th>
          <th data-sort="score" class="sort-active">Score</th>
          <th>Relevance</th>
        </tr>
      </thead>
      <tbody>
        {rows}
      </tbody>
    </table>

    <h2>Details</h2>
    {details}
  </div>

  <script>
    (function() {{
      var table = document.getElementById('rank-table');
      var tbody = table.querySelector('tbody');
      var headers = table.querySelectorAll('th[data-sort]');
      var state = {{ key: 'score', dir: -1 }};

      function rowsArray() {{ return Array.prototype.slice.call(tbody.querySelectorAll('tr.rank-row')); }}

      function sortBy(key, dir) {{
        var rows = rowsArray();
        rows.sort(function(a, b) {{
          var av, bv;
          if (key === 'score') {{
            av = parseFloat(a.dataset.score); bv = parseFloat(b.dataset.score);
          }} else if (key === 'name') {{
            av = a.dataset.name.toLowerCase(); bv = b.dataset.name.toLowerCase();
            return dir * (av < bv ? -1 : av > bv ? 1 : 0);
          }} else {{
            av = rows.indexOf(a); bv = rows.indexOf(b);
          }}
          return dir * (av - bv);
        }});
        rows.forEach(function(r) {{ tbody.appendChild(r); }});
      }}

      headers.forEach(function(th) {{
        th.addEventListener('click', function() {{
          var key = th.dataset.sort;
          state.dir = (state.key === key) ? -state.dir : -1;
          state.key = key;
          headers.forEach(function(hh) {{ hh.classList.remove('sort-active'); }});
          th.classList.add('sort-active');
          sortBy(key, state.dir);
        }});
      }});

      // Filter box: hide non-matching rows by transcript name; show a
      // "no matches" placeholder row when the filter empties the table.
      var filterInput = document.getElementById('transcript-filter');
      var noMatchRow = document.createElement('tr');
      noMatchRow.innerHTML = '<td class="no-matches" colspan="4">No transcripts match your filter.</td>';
      filterInput.addEventListener('input', function() {{
        var query = filterInput.value.trim().toLowerCase();
        var rows = rowsArray();
        var anyVisible = false;
        rows.forEach(function(r) {{
          var match = r.dataset.name.toLowerCase().indexOf(query) !== -1;
          r.style.display = match ? '' : 'none';
          if (match) anyVisible = true;
        }});
        if (!anyVisible && !tbody.contains(noMatchRow)) {{
          tbody.appendChild(noMatchRow);
        }} else if (anyVisible && tbody.contains(noMatchRow)) {{
          tbody.removeChild(noMatchRow);
        }}
      }});

      // Clicking a transcript name jumps to its detail section and expands it
      // (plain anchor scrolling doesn't set the `open` attribute on <details>).
      document.querySelectorAll('.col-name a').forEach(function(a) {{
        a.addEventListener('click', function() {{
          var id = a.getAttribute('href').slice(1);
          var el = document.getElementById(id);
          if (el) el.open = true;
        }});
      }});
    }})();
  </script>
</body>
</html>
"""


# --------------------------------------------------------------------------
# Entry points
# --------------------------------------------------------------------------

def write_html(report: BatchReport, out_path: str, top_locations: int = 5) -> str:
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(render_html(report, top_locations=top_locations))
    return out_path


def write_html_from_data(
    theme: str,
    transcripts: dict[str, list[tuple[str, dict]]],
    out_path: str,
    top_locations: int = 5,
) -> str:
    """Build and write ``report.html`` directly from in-memory run payloads
    (see ``build_report``), without touching disk for input."""
    report = build_report(theme, transcripts)
    return write_html(report, out_path, top_locations=top_locations)


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Regenerate report.html from the internal analysis cache"
    )
    parser.add_argument("--cache", required=True, help="Path to the .raw_results.json cache file")
    parser.add_argument("--out", default=None, help="Output .html path (defaults to report.html next to the cache file)")
    parser.add_argument("--top-locations", type=int, default=5, help="Max locations to show per transcript")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> None:
    args = parse_args(argv)
    out_path = args.out or os.path.join(os.path.dirname(args.cache) or ".", "report.html")
    try:
        theme, transcripts = read_cache(args.cache)
    except (FileNotFoundError, OSError) as exc:
        print(str(exc), file=sys.stderr)
        sys.exit(1)
    report = build_report(theme, transcripts)
    write_html(report, out_path, top_locations=args.top_locations)
    print(f"[html] {out_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile transcript_theme_analyzer/word_report.py
"""Generates a single consolidated Word (.docx) document with every relevant
excerpt from every transcript that matched the search theme:

- Document title = the search theme.
- An overview table (transcript × relevance score, in ranked order) right
  after the title, with clickable links that jump to each transcript's
  section further down.
- One section per transcript with at least one located excerpt, headed
  ``Transcript: <name>`` so an excerpt can always be traced back to its
  source. One sub-heading per excerpt, auto-generated from that excerpt's
  own context rather than just the search keyword.

Usage (regenerate later from the internal cache, without re-running analysis):
    python -m transcript_theme_analyzer.word_report --cache results/.raw_results.json
"""
from __future__ import annotations

import argparse
import os
import sys

from docx import Document
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Pt, RGBColor

from .cache import read_cache
from .report import BatchReport, TranscriptResult, build_report

_LINK_COLOR = RGBColor(0x25, 0x6A, 0xBF)  # same blue used for html links/bars
_FAILED_COLOR = RGBColor(0xD0, 0x3B, 0x3B)  # same red used for the html FAILED badge


def _add_horizontal_rule(document: Document) -> None:
    """A bottom-bordered empty paragraph, python-docx's standard recipe for
    a horizontal rule (Word has no native <hr> element)."""
    p = document.add_paragraph()
    p_pr = p._p.get_or_add_pPr()
    p_bdr = OxmlElement("w:pBdr")
    bottom = OxmlElement("w:bottom")
    bottom.set(qn("w:val"), "single")
    bottom.set(qn("w:sz"), "6")
    bottom.set(qn("w:space"), "1")
    bottom.set(qn("w:color"), "auto")
    p_bdr.append(bottom)
    p_pr.append(p_bdr)


def _add_bookmark(paragraph, name: str, bookmark_id: int) -> None:
    start = OxmlElement("w:bookmarkStart")
    start.set(qn("w:id"), str(bookmark_id))
    start.set(qn("w:name"), name)
    end = OxmlElement("w:bookmarkEnd")
    end.set(qn("w:id"), str(bookmark_id))
    paragraph._p.insert(0, start)
    paragraph._p.append(end)


def _add_internal_link(paragraph, text: str, bookmark_name: str) -> None:
    """Appends a clickable link to ``paragraph`` that jumps to ``bookmark_name``
    elsewhere in the same document (an anchor-only hyperlink — no external
    relationship, so python-docx's public hyperlink-reading API won't see it,
    but Word/LibreOffice render and follow it like any other link)."""
    hyperlink = OxmlElement("w:hyperlink")
    hyperlink.set(qn("w:anchor"), bookmark_name)

    run_element = OxmlElement("w:r")
    r_pr = OxmlElement("w:rPr")
    color = OxmlElement("w:color")
    color.set(qn("w:val"), "256ABF")
    r_pr.append(color)
    underline = OxmlElement("w:u")
    underline.set(qn("w:val"), "single")
    r_pr.append(underline)
    run_element.append(r_pr)

    text_element = OxmlElement("w:t")
    text_element.text = text
    run_element.append(text_element)

    hyperlink.append(run_element)
    paragraph._p.append(hyperlink)


def _add_excerpt(document: Document, location: dict) -> None:
    # The excerpt heading is generated from the location's own context
    # summary (produced by the analysis pass for that specific passage), not
    # from the search theme, so headings differ across excerpts even within
    # the same transcript.
    heading_text = (location.get("context_summary") or "").strip().rstrip(".")
    document.add_heading(heading_text or "Relevant excerpt", level=3)

    meta_bits = [b for b in (location.get("timestamp"), location.get("speaker")) if b]
    if meta_bits:
        meta_p = document.add_paragraph(" · ".join(meta_bits))
        for run in meta_p.runs:
            run.italic = True
            run.font.size = Pt(9)

    excerpt = (location.get("excerpt") or "").strip()
    p = document.add_paragraph()
    p.add_run(f"“{excerpt}”").italic = True


def _matching_transcripts(report: BatchReport, min_score: int | None) -> list[TranscriptResult]:
    return [
        t for t in report.transcripts
        if t.best_model is not None
        and t.best_model.locations
        and (min_score is None or (t.best_score or 0) >= min_score)
    ]


def _add_overview_table(document: Document, report: BatchReport, matched_ids: set[int]) -> None:
    heading = document.add_paragraph()
    run = heading.add_run("Overview")
    run.bold = True
    run.font.size = Pt(13)

    table = document.add_table(rows=1, cols=2)
    table.style = "Light Grid Accent 1"
    header_cells = table.rows[0].cells
    for cell, text in zip(header_cells, ("Transcript", "Relevance Score")):
        cell.paragraphs[0].add_run(text).bold = True

    for i, t in enumerate(report.transcripts):
        row = table.add_row()
        name_cell, score_cell = row.cells

        if id(t) in matched_ids:
            _add_internal_link(name_cell.paragraphs[0], t.name, f"t{i}")
        else:
            suffix = "" if t.best_model is not None else " (failed)"
            name_cell.paragraphs[0].add_run(f"{t.name}{suffix}")

        score_run = score_cell.paragraphs[0].add_run(
            str(t.best_score) if t.best_score is not None else "FAILED"
        )
        if t.best_score is None:
            score_run.font.color.rgb = _FAILED_COLOR


def build_docx(report: BatchReport, min_score: int | None = None) -> Document:
    document = Document()
    document.add_heading(report.theme, level=1)
    document.add_paragraph(f"Generated {report.generated_at}")

    matched = _matching_transcripts(report, min_score)
    matched_ids = {id(t) for t in matched}
    _add_overview_table(document, report, matched_ids)

    if not matched:
        document.add_paragraph("No transcripts contained relevant excerpts for this theme.")
        return document

    first = True
    for i, t in enumerate(report.transcripts):
        if id(t) not in matched_ids:
            continue
        if not first:
            _add_horizontal_rule(document)
        first = False

        heading = document.add_heading(f"Transcript: {t.name}", level=2)
        _add_bookmark(heading, f"t{i}", i)
        for location in t.best_model.locations:
            _add_excerpt(document, location)

    return document


def write_docx(report: BatchReport, out_path: str, min_score: int | None = None) -> str:
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    build_docx(report, min_score=min_score).save(out_path)
    return out_path


def write_docx_from_data(
    theme: str,
    transcripts: dict[str, list[tuple[str, dict]]],
    out_path: str,
    min_score: int | None = None,
) -> str:
    """Build and save the Word document directly from in-memory run payloads
    (same shape ``report.write_html_from_data`` takes) — no disk round-trip."""
    report = build_report(theme, transcripts)
    return write_docx(report, out_path, min_score=min_score)


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Regenerate report.docx from the internal analysis cache"
    )
    parser.add_argument("--cache", required=True, help="Path to the .raw_results.json cache file")
    parser.add_argument("--out", default=None, help="Output .docx path (defaults to report.docx next to the cache file)")
    parser.add_argument(
        "--min-score", type=int, default=None,
        help="Only include transcripts whose best score is >= this value (default: include any transcript with at least one located excerpt)",
    )
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> None:
    args = parse_args(argv)
    out_path = args.out or os.path.join(os.path.dirname(args.cache) or ".", "report.docx")
    try:
        theme, transcripts = read_cache(args.cache)
    except (FileNotFoundError, OSError) as exc:
        print(str(exc), file=sys.stderr)
        sys.exit(1)
    report = build_report(theme, transcripts)
    write_docx(report, out_path, min_score=args.min_score)
    print(f"[docx] {out_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile transcript_theme_analyzer/cli.py
"""CLI: run one theme against one or many .txt/.docx transcripts and one or
many models, producing exactly two output files: report.html and report.docx.

Usage (single transcript):
    python -m transcript_theme_analyzer.cli --transcript path.docx --theme "the glory of God" \\
        --models gpt-4.1 anthropic/claude-sonnet-5

Usage (whole folder of transcripts against the same theme):
    python -m transcript_theme_analyzer.cli --transcript-dir transcripts/ --theme "forgiveness" \\
        --models anthropic/claude-sonnet-5
"""
from __future__ import annotations

import argparse
import asyncio
import glob
import os
import sys
import time

from .analyzer import analyze
from .cache import write_cache
from .client import make_client
from .config import load_config
from .loader import detect_format, load_transcript_text
from .report import write_html_from_data
from .word_report import write_docx_from_data

DEFAULT_GLOB_PATTERNS = ["*.txt", "*.docx"]


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Transcript theme-relevance analyzer")
    source = parser.add_mutually_exclusive_group(required=True)
    source.add_argument("--transcript", help="Path to a single transcript file (.txt or .docx)")
    source.add_argument(
        "--transcript-dir",
        help="Directory of transcript files (.txt and/or .docx) to run the same theme against",
    )
    parser.add_argument(
        "--glob",
        default=None,
        help="Filename pattern to match inside --transcript-dir (default: *.txt and *.docx)",
    )
    parser.add_argument("--theme", required=True, help="Theme to analyze for (word, phrase, or paragraph)")
    parser.add_argument(
        "--models",
        nargs="+",
        default=None,
        help="One or more model identifiers to run (defaults to config default_model)",
    )
    parser.add_argument(
        "--out-dir",
        default="results",
        help="Directory to write the report (report.html, report.docx) into",
    )
    return parser.parse_args(argv)


def _discover_transcript_paths(transcript_dir: str, glob_pattern: str | None) -> list[str]:
    patterns = [glob_pattern] if glob_pattern else DEFAULT_GLOB_PATTERNS
    paths = {p for pattern in patterns for p in glob.glob(os.path.join(transcript_dir, pattern))}
    return sorted(paths)


def _validate_formats_upfront(paths: list[str]) -> None:
    """Check every path's format before running any (paid) analysis, so a
    bad/unsupported file is caught immediately rather than after other
    transcripts in the same batch have already been analyzed."""
    errors = []
    for path in paths:
        try:
            detect_format(path)
        except ValueError as exc:
            errors.append(str(exc))
    if errors:
        print("Cannot proceed — some input files are unsupported/invalid:", file=sys.stderr)
        for err in errors:
            print(f"  - {err}", file=sys.stderr)
        sys.exit(1)


async def run_one_model(transcript: str, theme: str, model: str, config) -> dict:
    client = make_client(config)
    start = time.monotonic()
    result, stats = await analyze(client, model, theme, transcript, config)
    elapsed = time.monotonic() - start
    payload = result.model_dump()
    payload["_meta"] = {
        "elapsed_seconds": round(elapsed, 2),
        "usage": stats.summary(),
    }
    return payload


async def run_all(transcript: str, theme: str, models: list[str]) -> list[tuple[str, dict]]:
    """Run every model against one transcript. Returns a list of
    ``(model, raw_result_payload)`` pairs, in the same order as ``models`` —
    a failed run's payload is ``{"model": ..., "error": ...}``.
    """
    config = load_config()

    results = await asyncio.gather(
        *[run_one_model(transcript, theme, model, config) for model in models],
        return_exceptions=True,
    )

    raw_results: list[tuple[str, dict]] = []
    for model, result in zip(models, results):
        if isinstance(result, Exception):
            print(f"[{model}] FAILED: {result}", file=sys.stderr)
            raw_results.append((model, {"model": model, "error": str(result)}))
            continue
        print(f"[{model}] score={result['relevance_score']}")
        raw_results.append((model, result))
    return raw_results


def _write_outputs(theme: str, raw_by_transcript: dict[str, list[tuple[str, dict]]], out_dir: str) -> None:
    cache_path = write_cache(theme, raw_by_transcript, out_dir)
    print(f"[cache] {cache_path} (internal — not a final output)")

    html_path = write_html_from_data(theme, raw_by_transcript, os.path.join(out_dir, "report.html"))
    print(f"[report:html] {html_path}")

    docx_path = write_docx_from_data(theme, raw_by_transcript, os.path.join(out_dir, "report.docx"))
    print(f"[report:docx] {docx_path}")


async def run_batch(
    transcript_paths: list[str], theme: str, models: list[str], out_dir: str
) -> None:
    raw_by_transcript: dict[str, list[tuple[str, dict]]] = {}
    for path in transcript_paths:
        stem = os.path.splitext(os.path.basename(path))[0]
        print(f"\n=== {stem} ===")
        try:
            transcript = load_transcript_text(path)
        except (ValueError, OSError) as exc:
            print(f"[{stem}] FAILED to read transcript: {exc}", file=sys.stderr)
            raw_by_transcript[stem] = [
                (model, {"model": model, "error": f"could not read transcript: {exc}"})
                for model in models
            ]
            continue
        raw_by_transcript[stem] = await run_all(transcript, theme, models)

    print()
    _write_outputs(theme, raw_by_transcript, out_dir)


def main(argv: list[str] | None = None) -> None:
    args = parse_args(argv)
    config = load_config()
    models = args.models or [config.default_model]

    if args.transcript_dir:
        paths = _discover_transcript_paths(args.transcript_dir, args.glob)
        if not paths:
            pattern_desc = args.glob or " or ".join(DEFAULT_GLOB_PATTERNS)
            print(
                f"No files matching {pattern_desc!r} found in {args.transcript_dir}",
                file=sys.stderr,
            )
            sys.exit(1)
        _validate_formats_upfront(paths)
        asyncio.run(run_batch(paths, args.theme, models, args.out_dir))
        return

    transcript = load_transcript_text(args.transcript)
    transcript_stem = os.path.splitext(os.path.basename(args.transcript))[0]
    raw_results = asyncio.run(run_all(transcript, args.theme, models))
    _write_outputs(args.theme, {transcript_stem: raw_results}, args.out_dir)


if __name__ == "__main__":
    main()


## Install dependencies

In [ ]:
!pip install -q "openai>=1.50.0" "pydantic>=2.6.0" "python-docx>=1.1.0"


## Configure your LLM provider + API key

This cell auto-detects where it's running:
- **Colab** — reads from Colab's Secrets manager (the key icon in the left
  sidebar). Add a secret named `OPENROUTER_API_KEY` there first, and make
  sure "Notebook access" is toggled on for it.
- **Kaggle** — reads from Kaggle's Secrets add-on (Add-ons → Secrets). Add a
  secret named `OPENROUTER_API_KEY` there first.
- **Neither found** — falls back to a masked prompt (`getpass`) so it still
  works, just not persisted anywhere.

Switching to OpenAI direct instead of OpenRouter: change the three
`LLM_*`/`OPENAI_API_KEY` lines below to match `.env.example` in the repo.


In [ ]:
import getpass
import os


def set_secret_env(env_var: str, prompt: str | None = None) -> None:
    if os.environ.get(env_var):
        return

    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(env_var)
    except Exception:
        pass

    if not value:
        try:
            from kaggle_secrets import UserSecretsClient  # type: ignore
            value = UserSecretsClient().get_secret(env_var)
        except Exception:
            pass

    if not value:
        value = getpass.getpass(prompt or f"Enter {env_var}: ")

    os.environ[env_var] = value


set_secret_env("OPENROUTER_API_KEY")

os.environ.setdefault("LLM_PROVIDER", "openrouter")
os.environ.setdefault("LLM_BASE_URL", "https://openrouter.ai/api/v1")
os.environ.setdefault("LLM_DEFAULT_MODEL", "anthropic/claude-sonnet-5")
os.environ.setdefault("CHUNK_SIZE_TOKENS", "12000")
os.environ.setdefault("CHUNK_OVERLAP_TOKENS", "800")
os.environ.setdefault("SINGLE_PASS_TOKEN_LIMIT", "20000")
os.environ.setdefault("MAX_CONCURRENT_CHUNKS", "8")
os.environ.setdefault("LLM_MAX_RETRIES", "5")

print("LLM_PROVIDER:", os.environ["LLM_PROVIDER"])
print("LLM_DEFAULT_MODEL:", os.environ["LLM_DEFAULT_MODEL"])
print("API key set:", bool(os.environ.get("OPENROUTER_API_KEY")))


## ⚠️ Kaggle only: enable internet access

Kaggle notebooks have internet access **off by default**. Since every
analysis call needs to reach the LLM API, you must turn it on before
running: **Settings (right sidebar) → Internet → On** (requires phone
verification on your Kaggle account, one-time). Colab has internet on by
default — nothing to do there.


## Get your transcripts onto the notebook — Colab

Run the cell below **on Colab**. It uploads files directly into a local
`transcripts/` folder via a browser file picker. For a folder that persists
across sessions instead, mount Google Drive (commented out below) and point
`TRANSCRIPT_DIR` at a path under `/content/drive/MyDrive/...`.


In [ ]:
import os

TRANSCRIPT_DIR = "transcripts"

try:
    from google.colab import files  # type: ignore

    os.makedirs(TRANSCRIPT_DIR, exist_ok=True)
    print(f"Select your .txt/.docx transcript files (uploading into {TRANSCRIPT_DIR}/):")
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(os.path.join(TRANSCRIPT_DIR, name), "wb") as f:
            f.write(content)
    print(f"Uploaded {len(uploaded)} file(s) into {TRANSCRIPT_DIR}/")

    # Alternative: a persistent folder via Google Drive instead of a one-off upload.
    # from google.colab import drive
    # drive.mount("/content/drive")
    # TRANSCRIPT_DIR = "/content/drive/MyDrive/path/to/your/transcripts"
except ImportError:
    print("Not running on Colab — skipping this cell (see the Kaggle cell below instead).")


## Get your transcripts onto the notebook — Kaggle

On Kaggle: click **"Add Input"** (top right) → **Upload Dataset** → upload
your `.txt`/`.docx` transcript files. Once attached, they appear read-only
under `/kaggle/input/<your-dataset-name>/`. Set `KAGGLE_TRANSCRIPT_DIR`
below to match, and it'll override `TRANSCRIPT_DIR` from the Colab cell
above when running on Kaggle.


In [ ]:
import os

KAGGLE_TRANSCRIPT_DIR = "/kaggle/input/<your-dataset-name>"  # <-- change this

if os.path.isdir("/kaggle/input"):
    TRANSCRIPT_DIR = KAGGLE_TRANSCRIPT_DIR
    print(f"Running on Kaggle — TRANSCRIPT_DIR set to {TRANSCRIPT_DIR}")
else:
    print("Not running on Kaggle — leaving TRANSCRIPT_DIR as set by the Colab cell above.")


## Run the analysis

Edit `THEME` and `MODELS` below, then run this cell. It calls the same
`transcript_theme_analyzer.cli` you'd run locally — see the repo's README
for the full flag reference (`--glob`, multiple `--models`, etc.).


In [ ]:
THEME = "the glory of God"  # <-- change this to your search theme
MODELS = "anthropic/claude-sonnet-5"  # space-separated if running more than one model
OUT_DIR = "results"

!python -m transcript_theme_analyzer.cli --transcript-dir {TRANSCRIPT_DIR} --theme "{THEME}" --models {MODELS} --out-dir {OUT_DIR}


## View and download the results

Two final outputs land in `OUT_DIR`: `report.html` (rendered inline below)
and `report.docx`. On Colab, `report.docx` downloads directly. On Kaggle,
everything under `/kaggle/working` is automatically saved as notebook
output and downloadable from the Output tab after you save a version.


In [ ]:
from IPython.display import HTML, display

with open(f"{OUT_DIR}/report.html", encoding="utf-8") as f:
    display(HTML(f.read()))


In [ ]:
try:
    from google.colab import files  # type: ignore
    files.download(f"{OUT_DIR}/report.docx")
except ImportError:
    print(f"Not on Colab — find {OUT_DIR}/report.docx in the Output tab (Kaggle) "
          f"or the file browser on the left (other environments).")
